# C1 · 캡스톤 1: PageRank (Capstone 1: PageRank as a Markov Chain)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §1.4 (정상분포), §1.5 (극한 거동), §1.7 (수렴 증명), §1.8–1.9 (출구 분포·시간) · Lawler 2e Ch. 1 (유한 사슬: 불변확률과 수렴) · Norris §1.7–1.8 · Langville & Meyer (2006) Ch. 4, Thm 4.1 · Haveliwala & Kamvar (2003) Thm 1–2 |
| 선수 노트북 | 01d (정상분포 세 방법), 01e (수렴 정리·TV 거리·$\lambda_2$), 01f (사슬 설계 관점), 01c (기본행렬·도달시간), 01r, 00b (MC 오차) |
| 예상 소요 | 120분 |
| 상태 | draft |

## 프로젝트 브리프 · 합격 기준 · 단계

**브리프.** 7개 노드짜리 작은 웹 그래프를 마르코프 사슬로 모델링하여 PageRank를 구하고, damping $d$·매달린(dangling) 노드·랭크 싱크(rank sink)가 정상분포와 수렴 속도에 미치는 영향을 정량화한다. 그래프는 고정이다:

$$\texttt{edges} = \{0{:}[1,2,3],\ 1{:}[0,2,6],\ 2{:}[0],\ 3{:}[4],\ 4{:}[5],\ 5{:}[4],\ 6{:}[\,]\}$$

노드 6은 매달린 노드(나가는 링크 없음), $\{4,5\}$는 닫힌 2-사이클(랭크 싱크)이다. 기본값은 $d=0.85$, 텔레포트 분포 $v$ = 균등.

**합격 기준 — 이 그림/숫자가 나오면 성공.**

1. $\pi(0.85) \approx [0.1033,\ 0.0559,\ 0.0717,\ 0.0559,\ 0.3483,\ 0.3226,\ 0.0424]$가 세 가지 방법(선형계·고유벡터·거듭제곱)과 선형계 형태, spkit `pagerank`에서 $10^{-10}$ 이내로 일치한다.
2. 거듭제곱 반복의 $L^1$ 오차가 반로그 그림에서 기울기 $\log d$의 직선(비율 0.85)을 따르고, $|\lambda_2(G)| = d = 0.85$가 수치로 확인된다.
3. 랜덤 서퍼 20,000명(FAST 2,000명)의 60걸음 후 위치 분포가 모든 노드에서 $\pi$와 4 SE 이내다.
4. $d\to1$에서 질량이 $\{4,5\}$로 몰리고($[0,0,0,0,\tfrac12,\tfrac12,0]$), $d\to0$에서 균등으로 가는 민감도 그림이 나온다.
5. 매달린 노드를 고치지 않으면 총질량이 300회 반복 후 약 $6.8\times10^{-5}$로 새어 나가는 것을 재현한다.

**단계.**

| 단계 | 내용 | 재사용 | 산출물 |
|---|---|---|---|
| Task 1 | 그래프 → 행렬 $A, S, G$, 구조 진단 | 01b | 그래프 그림, 류·주기 |
| Task 2 | 정상분포 세 방법 + 선형계 형태 | 01d | 합격 1, 막대그림, 순위표 |
| Task 3 | 거듭제곱 반복의 오차와 스펙트럼 | 01e | 합격 2, 반로그 그림 |
| Task 4 | 랜덤 서퍼 몬테카를로 | 00b, 01c | 합격 3, Compare 표 |
| Task 5 | 민감도: damping·개인화·누수 | 01f | 합격 4·5 |

스트레치 목표는 연습문제 S1–S3에 있다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import (is_stochastic, communicating_classes, classify_states, pagerank,
                          stationary, stationary_eig, stationary_power, absorption,
                          simulate_chain, empirical_distribution, hitting_times, tv_distance)

SEED, rng = rng_for("C1")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (01r/01d) 유한 상태 기약 사슬에서 정상분포의 존재와 유일성은 무엇이 보장하는가? 정상분포를 구하는 세 가지 방법을 말하라.

<details><summary>정답</summary>

유한 + 기약이면 유일한 정상분포가 존재한다(Perron–Frobenius). (i) $\pi P = \pi,\ \sum\pi = 1$ 선형계(방정식 하나를 정규화로 교체), (ii) 고유값 1의 왼쪽 고유벡터, (iii) 긴 경로의 상태별 점유 빈도(에르고딕 정리).

</details>

**Q2.** (01r/01e) 수렴 정리의 두 가정은 무엇이며 각각이 깨지면 무슨 일이 생기는가?

<details><summary>정답</summary>

기약성과 비주기성. 기약이 아니면 시작 상태에 따라 극한이 달라진다(닫힌 류가 여럿). 주기적이면 $P^n(x,y)$가 진동하고 Cesàro 평균만 수렴한다 — 이 노트북의 2-사이클 $\{4,5\}$가 그 예다.

</details>

**Q3.** (01r/01e) 가역 사슬에서 TV 거리는 어떤 속도로 줄어드는가? $\lambda_2$는 무엇인가?

<details><summary>정답</summary>

$\mathrm{TV}(P^n(x,\cdot),\pi) \le C|\lambda_2|^n$. $\lambda_2$는 절댓값이 두 번째로 큰 고유값이고 스펙트럴 갭 $1-|\lambda_2|$가 혼합 속도를 정한다. 가역이 아니면 고유값이 복소수일 수 있지만 $|\lambda_2|<1$이면 여전히 기하급수적으로 수렴한다.

</details>

**Q4.** (01g, 두 노트북 전) 골턴–왓슨 과정의 소멸확률은 어떻게 구하며 $m \le 1$이면 어떻게 되는가?

<details><summary>정답</summary>

자손 pgf $f$의 $[0,1]$에서 가장 작은 고정점 $s = f(s)$. $m \le 1$이면(자손이 항상 1인 퇴화 경우 제외) 소멸확률 1.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **인접행렬** $A$: $A_{ij}=1$이면 페이지 $i$가 $j$로 링크한다. 출차수(out-degree) $\mathrm{out}_i = \sum_j A_{ij}$. **링크 행렬** $S_{ij} = A_{ij}/\mathrm{out}_i$ ($\mathrm{out}_i>0$); 매달린 노드($\mathrm{out}_i=0$)의 행은 균등 $1/n$으로 채운다(dangling fix).
- **구글 행렬(Google matrix)** $G = dS + (1-d)\mathbf{1}v^\top$. damping $d\in(0,1)$(기본 0.85), 텔레포트 분포 $v$(기본 균등 $1/n$; 개인화 PageRank는 $v = e_i$ 등). **PageRank 벡터** $\pi$는 $G$의 정상분포다: $\pi G = \pi,\ \sum\pi = 1$.
- **랜덤 서퍼(random surfer)**: 각 걸음마다 확률 $d$로 현재 페이지의 링크 중 하나를 균등하게 따라가고, 확률 $1-d$로 $v$에서 새 페이지로 순간이동(텔레포트)한다. 매달린 페이지에서는 항상 순간이동한다. 이 서퍼의 위치가 정확히 전이행렬 $G$를 가진 마르코프 사슬이다.
- **랭크 싱크(rank sink)**: $S$의 닫힌 류 중 나가는 링크가 없는 부분그래프(본 그래프의 $\{4,5\}$). $d=1$이면 모든 질량이 그곳에 갇힌다.
- **거듭제곱 반복(power iteration)**: $x_{k+1} = x_k G$, $x_0 = \mathbf{1}/n$. 오차 $\|x_k - \pi\|_1 = 2\,\mathrm{TV}(x_k,\pi)$.
- **선형계 형태**: $\pi = (1-d)\,v^\top (I - dS)^{-1}$ (균등 $v$이면 $\frac{1-d}{n}\mathbf{1}^\top(I-dS)^{-1}$).

### 2.2 정리 1 — 구글 행렬의 기약·비주기성과 유일 정상분포 (Durrett 3e §1.4–1.5)

> $d<1$이고 $v$의 모든 성분이 양수이면 $G$의 모든 원소가 $\ge (1-d)\min_j v_j > 0$이므로 $G$는 기약·비주기이다. 따라서 유일한 정상분포 $\pi$가 존재하고 모든 초기분포에서 $x_k \to \pi$이다.

**가정이 왜 필요한가**

- $d<1$: $d=1$이면 $G=S$가 되어 (i) 닫힌 류가 둘 이상이면 정상분포가 유일하지 않고, (ii) 닫힌 류가 주기적이면 거듭제곱 반복이 수렴하지 않는다. 본 그래프는 (ii)에 해당한다: 닫힌 류는 $\{4,5\}$ 하나뿐이라 $\pi_S = [0,0,0,0,\tfrac12,\tfrac12,0]$은 유일하지만, 주기 2라 $x_k$는 노드 4·5 위에서 0.435/0.565 사이를 진동하고, 기약이 아니라 $\{4,5\}$ 밖 상태의 질량은 모두 사라진다(E3).
- $v>0$: $v$에 0이 있으면 양의 행렬이 아니게 되어 기약성을 그래프 구조에서 따로 확인해야 한다(개인화 PageRank의 함정).

*증명 (인용).* 모든 원소가 양수인 행렬은 자명하게 기약·비주기(모든 $x\to y$가 한 걸음에 가능, $P(x,x)>0$). 나머지는 01e의 수렴 정리(Durrett 3e §1.5 Thm 1.19, 증명은 §1.7). 본 노트북에서는 정리 2로 더 강한(정량적) 결과를 직접 증명한다.

### 2.3 정리 2 — $L^1$ 수축 정리 (거듭제곱 반복의 수렴 속도)

> 임의의 두 확률벡터 $x, y$에 대해 $\|(x-y)G\|_1 \le d\,\|x-y\|_1$. 따라서 $\|x_k-\pi\|_1 \le d^k\|x_0-\pi\|_1 \le 2d^k$, 즉 $\mathrm{TV}(x_k,\pi) \le d^k$.

**가정이 왜 필요한가**

- $x-y$의 성분합이 0: 이 덕분에 $(x-y)\mathbf{1}v^\top = 0$이 되어 텔레포트 항이 사라진다. 확률벡터가 아닌 일반 벡터에는 성립하지 않는다.
- $S$가 행확률(행합 1, 비음): $\|zS\|_1 \le \|z\|_1$의 근거. 매달린 노드를 고치지 않으면 $S$의 행합이 1보다 작아 질량이 새고 정리의 등식 구조가 깨진다.

<details><summary>증명</summary>

$z = x - y$라 하면 $z\mathbf{1} = \sum_i z_i = 1 - 1 = 0$. 그러므로

$$zG = d\,zS + (1-d)(z\mathbf{1})v^\top = d\,zS.$$

이제 $S$가 비음이고 행합이 1이므로 삼각부등식으로

$$\|zS\|_1 = \sum_j\Big|\sum_i z_iS_{ij}\Big| \le \sum_i|z_i|\sum_j S_{ij} = \sum_i |z_i| = \|z\|_1 .$$

따라서 $\|zG\|_1 = d\|zS\|_1 \le d\|z\|_1$. 마지막으로 $\pi G = \pi$이므로 $x_k - \pi = (x_{k-1}-\pi)G = \cdots = (x_0-\pi)G^k$이고, 각 단계에 위 부등식을 적용하면 $\|x_k-\pi\|_1 \le d^k\|x_0-\pi\|_1$. 두 확률벡터의 $L^1$ 거리는 최대 2이므로 $\le 2d^k$. $\|\cdot\|_1 = 2\,\mathrm{TV}$이므로 $\mathrm{TV}(x_k,\pi)\le d^k$. $\blacksquare$

본 그래프의 수치: $\|x_0-\pi\|_1 = 0.770$ (균등 $x_0$), 그러므로 $\|x_k-\pi\|_1 < 10^{-6}$은 늦어도 $k = \lceil\log(10^{-6}/0.770)/\log 0.85\rceil = 84$회에 보장된다. 실제로는 72회에 도달한다(Task 3).

</details>

### 2.4 정리 3 — 구글 행렬의 스펙트럼 (Haveliwala–Kamvar 2003 Thm 1–2; Langville & Meyer Thm 4.1)

> $S$의 고유값이 $\{1, \lambda_2, \dots, \lambda_n\}$이면 $G = dS + (1-d)\mathbf{1}v^\top$의 고유값은 $\{1, d\lambda_2, \dots, d\lambda_n\}$이다. 특히 $|\lambda_2(G)| \le d$이고, $S$에 닫힌 류가 둘 이상이거나 주기적 닫힌 류가 있으면($|\lambda_2(S)|=1$) $|\lambda_2(G)| = d$이다.

**가정이 왜 필요한가**

- $S\mathbf{1} = \mathbf{1}$ (행확률): 닮음변환 $Q=[\mathbf{1}\ X]$에서 첫 열이 고유벡터가 되어 블록 상삼각이 된다.
- $v^\top\mathbf{1} = 1$ (확률벡터): 첫 대각원소가 $d + (1-d) = 1$이 된다.

*증명 (스케치).* 가역 $Q = [\mathbf{1}\ X]$를 잡고 $Q^{-1} = \begin{pmatrix} y^\top \\ Y^\top\end{pmatrix}$라 쓰면 $Q^{-1}Q = I$에서 $y^\top\mathbf{1}=1$, $Y^\top\mathbf{1}=0$. $S\mathbf{1}=\mathbf{1}$이므로
$Q^{-1}SQ = \begin{pmatrix}1 & y^\top SX\\ 0 & Y^\top SX\end{pmatrix}$이고 $Y^\top SX$의 고유값이 $\lambda_2,\dots,\lambda_n$이다. 한편 $Q^{-1}\mathbf{1}v^\top Q = \begin{pmatrix}1\\0\end{pmatrix}(v^\top\mathbf{1},\ v^\top X) = \begin{pmatrix}1 & v^\top X\\ 0 & 0\end{pmatrix}$. 둘을 합치면
$Q^{-1}GQ = \begin{pmatrix}1 & *\\ 0 & d\,Y^\top SX\end{pmatrix}$ → 고유값 $1, d\lambda_2, \dots, d\lambda_n$. 블록 구조가 $v$에 의존하지 않으므로 **고유값은 텔레포트 분포와 무관**하다. 본 그래프에서는 $\{4,5\}$가 주기 2인 닫힌 류라 $S$의 고유값에 $-1$이 있고, 따라서 $G$의 고유값에 $-0.85$가 있다.

### 2.5 정리 4 — 선형계 형태

> $d<1$이면 $I-dS$는 가역이고 $\pi = (1-d)\,v^\top(I-dS)^{-1}$.

**가정이 왜 필요한가**

- $d<1$: $\rho(dS) = d < 1$이어야 $I - dS$가 가역(노이만 급수 $\sum_k d^kS^k$ 수렴).
- $\pi\mathbf{1} = 1$: $\pi\mathbf{1}v^\top = v^\top$로 단순화하는 데 필요.

<details><summary>증명</summary>

$\pi = \pi G = d\pi S + (1-d)(\pi\mathbf{1})v^\top = d\pi S + (1-d)v^\top$이므로 $\pi(I-dS) = (1-d)v^\top$. $S$의 스펙트럴 반지름은 1이므로 $\rho(dS) = d<1$이고 $I-dS$는 가역이며 노이만 급수 $(I-dS)^{-1} = \sum_{k\ge0}d^kS^k$가 수렴한다. 따라서

$$\pi = (1-d)\sum_{k\ge0}d^k\, v^\top S^k .$$

해석: 서퍼가 마지막으로 텔레포트한 뒤 $k$걸음 걸었을 확률이 $(1-d)d^k$이고, 그때의 위치 분포가 $v^\top S^k$이다. 즉 $\pi$는 '텔레포트 후 링크를 $k$번 따라간 경로'의 가중합 — 랜덤 서퍼 해석과 정확히 일치한다. $\blacksquare$

</details>

### 2.6 직관

PageRank는 '무한히 오래 서핑한 사람이 지금 어디 있을까'의 확률이다. 링크를 많이 받는 것보다 **'중요한 페이지가 자기 링크를 아껴서 주는' 것**이 크다: 노드 0은 링크를 두 곳(1, 2)에서 받지만 2-사이클 0→1→0, 0→2→0(과 3-사이클 0→1→2→0)을 도는 동안 노드 3을 거쳐 $\{4,5\}$로 질량을 흘리고, $\{4,5\}$는 서로만 가리켜 질량을 가둔다. 텔레포트는 수학적으로는 기약·비주기성을 강제하는 장치이고(정리 1), 동시에 수렴 속도를 $d$로 못 박는 장치다(정리 2): 그래프가 10억 노드여도 $d=0.85$, 목표 $10^{-6}$이면 90회 이하로 충분하다(일반 상계 $2d^k$); 본 그래프의 균등 $x_0$에서는 상계 84회, 실제 72회다(Task 3).

### 2.7 함정 (traps)

- 매달린 노드를 고치지 않으면(행합 0) 질량이 매 반복 새어 나간다: 300회 후 총합 $\approx 6.8\times10^{-5}$. 정규화해도 $[0.092, 0.049, 0.063, 0.049, 0.367, 0.344, 0.036]$처럼 참 $\pi$와 다르다.
- $d$는 최적화 대상이 아니라 **모델 가정**이다. $d\to1$: 랭크 싱크 $\{4,5\}$가 전부 흡수($\pi_4+\pi_5 \to 1$); $d\to0$: 균등. 순위 1·2위(4, 5)는 $d \ge 0.3$에서 안정하지만 $d$가 작으면 순위가 바뀐다($d=0.1$에서는 노드 0이 5를 앞선다).
- 수렴 속도는 그래프 크기가 아니라 $d$가 정한다(정리 2). $|\lambda_2(G)| = d|\lambda_2(S)|$는 이보다 **빠를 수** 있음을 말할 뿐이다(랭크 싱크가 없으면 $|\lambda_2(G)| < d$; 5→4 링크를 지운 변형에서는 0.338).
- 시간 평균(01d의 세 번째 방법)은 자기상관 때문에 SE를 바로 주지 않는다. 독립 서퍼 $n$명을 $k$걸음 뒤에 관측하면 iid → `mc_proportion`; 편향은 $\le d^k$ ($k=60$: $5.8\times10^{-5}$, SE 0.0034에 비해 무시).
- $\pi_1 = \pi_3$은 정확한 동률이다: 둘 다 노드 0에서만(1/3씩) 링크를 받으므로 $\pi_1 = d\pi_0/3 + (1-d)/7 = \pi_3$. 'in-degree가 같다'가 아니라 **'같은 곳에서 같은 비율로 받는다'**가 이유다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

## 3. Predict — 코드를 돌리기 전에 예측

예측을 먼저 적어 보세요. 정답은 뒤에서 코드가 계산합니다. 틀려도 좋습니다 — 나중에 왜 틀렸는지가 학습 포인트입니다.

In [ ]:
predictions = {
    # d=0.85에서 PageRank 1위 노드는? (in-degree가 2인 노드 0, 2, 4 중 고르세요)
    "pi_top_node": None,
    # 그 1위 노드의 값 pi_max는?
    "pi_max": None,
    # 허브 노드 0의 PageRank pi_0는?
    "pi_node0": None,
    # |lambda_2(G)|는 얼마인가? (힌트: S에 주기 2인 닫힌 류가 있다 → 정리 3)
    "lambda2_abs": None,
    # 균등 초기벡터에서 ||x_k - pi||_1 < 1e-6 이 되는 최소 반복 횟수 k는? (상계 d^k ||x_0 - pi||_1 로 예측)
    "n_iter_1e6": None,
    # 노드 0에서 출발한 서퍼가 노드 4에 처음 닿기까지 기대 걸음 수 E_0[T_4]는?
    "hit_time_0_to_4": None,
    # 노드 0에서 출발해 노드 0으로 처음 되돌아오는 기대 시간 E_0[T_0^+]는? (01d/04c: 1/pi_0)
    "return_time_0": None,
}

## 4. 단계별 작업 (Tasks) — Simulate

spkit에 없는 도우미를 먼저 한 셀에 정의한다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def google_matrix(A, d, v=None):
    """(G, S): S = 링크 행렬(매달린 행 → 균등), G = d S + (1-d) 1 v^T (v 기본 = 균등)."""
    A = np.asarray(A, float); n = A.shape[0]
    out = A.sum(axis=1)
    v = np.full(n, 1.0 / n) if v is None else np.asarray(v, float)
    S = np.where(out[:, None] > 0, A / np.where(out > 0, out, 1.0)[:, None], 1.0 / n)
    return d * S + (1 - d) * np.outer(np.ones(n), v), S

def linear_form_pagerank(S, d, v):
    """정리 4: pi = (1-d) v^T (I - dS)^{-1}  ←  (I - dS)^T y = v 를 풀어서 (역행렬 없이)."""
    n = S.shape[0]
    return (1 - d) * np.linalg.solve((np.eye(n) - d * S).T, v)

def power_iteration(G, x0, n_iter, pi):
    """x_{k+1} = x_k G. errs[k] = ||x_k - pi||_1 (k = 0..n_iter) 와 반복 벡터들을 반환."""
    xs = [np.asarray(x0, float)]
    for _ in range(n_iter):
        xs.append(xs[-1] @ G)
    xs = np.array(xs)
    return np.abs(xs - pi).sum(axis=1), xs

def first_return_times(paths, x):
    """각 경로에서 t >= 1 중 처음 paths[t] == x 인 t (없으면 inf)."""
    hit = paths[:, 1:] == x
    return np.where(hit.any(axis=1), hit.argmax(axis=1) + 1, np.inf)

def abs_eigs_sorted(P):
    """고유값 절댓값을 내림차순으로."""
    return np.sort(np.abs(np.linalg.eigvals(P)))[::-1]

### Task 1 — 그래프를 행렬로: $A$, $S$, $G$ 만들기와 구조 진단

먼저 그래프를 그린다(networkx 없이 matplotlib만). 주황 = 매달린 노드, 빨강 = 랭크 싱크.

In [ ]:
edges = {0: [1, 2, 3], 1: [0, 2, 6], 2: [0], 3: [4], 4: [5], 5: [4], 6: []}
n, d = 7, 0.85
A = np.zeros((n, n))
for i, js in edges.items():
    A[i, js] = 1.0
G, S = google_matrix(A, d)

def draw_graph(edges, pos, ax, dangling=(), sink=()):
    for i, js in edges.items():
        for j in js:
            ax.annotate("", xy=pos[j], xytext=pos[i], zorder=1,
                        arrowprops=dict(arrowstyle="->", lw=1.3, color="0.45", shrinkA=15,
                                        shrinkB=15, connectionstyle="arc3,rad=0.18"))
    for i, (x, y) in pos.items():
        c = "tab:red" if i in sink else ("tab:orange" if i in dangling else "tab:blue")
        ax.scatter([x], [y], s=650, c=c, zorder=3)
        ax.text(x, y, str(i), ha="center", va="center", color="white", fontsize=12, zorder=4)

pos = {0: (0, 0), 1: (-1.2, 1.0), 2: (-1.2, -1.0), 3: (1.3, 0), 4: (2.6, 0.8), 5: (2.6, -0.8), 6: (-2.6, 1.0)}
fig, ax = plt.subplots(figsize=(7, 3.6))
draw_graph(edges, pos, ax, dangling={6}, sink={4, 5})
ax.set(xlim=(-3.3, 3.3), ylim=(-1.6, 1.6), title="Web graph (7 nodes): orange = dangling, red = rank sink {4,5}")
ax.set_axis_off(); plt.show()

$S$와 $G$가 행확률인지, 그리고 류(communicating class)·주기 구조가 텔레포트로 어떻게 바뀌는지 확인한다.

In [ ]:
np.set_printoptions(precision=4, suppress=True)
print("S is stochastic:", is_stochastic(S), "  G is stochastic:", is_stochastic(G))
print("min(G) =", G.min(), " (>= (1-d)/n =", (1 - d) / n, ")")
cs, cg = classify_states(S), classify_states(G)
print("S: classes", communicating_classes(S), " closed:", [cs["classes"][k] for k in cs["recurrent"]],
      " periods:", cs["period"])
print("G: classes", cg["classes"], " closed:", [cg["classes"][k] for k in cg["recurrent"]], " periods:", cg["period"])
pi_spkit = pagerank(A, d)          # Task 2에서 대조할 spkit 결과
print("spkit pagerank(A, 0.85) =", pi_spkit)

$S$만 보면 류가 셋($\{0,1,2,6\}, \{3\}, \{4,5\}$)이고 닫힌 류는 주기 2인 $\{4,5\}$뿐이다(류 $\{3\}$의 주기 0은 사이클이 없는 일시 상태라는 뜻). $G$는 모든 원소가 $\ge (1-d)/n$이라 류 하나·주기 1 — 정리 1이 그대로 보인다.

### Task 2 — 정상분포 세 가지 방법 + 선형계 형태 (01d 재사용)

In [ ]:
v_unif = np.full(n, 1.0 / n)
methods = {
    "linear system  (stationary)": stationary(G),
    "left eigenvector (stationary_eig)": stationary_eig(G),
    "matrix power G^200 (stationary_power)": stationary_power(G, 200),
    "spkit pagerank(A, d)": pi_spkit,
    "(1-d) v (I - dS)^-1  (Thm 4)": linear_form_pagerank(S, d, v_unif),
}
pi = methods["linear system  (stationary)"]
for name, val in methods.items():
    print(f"{name:40s} {val}   max|diff| = {np.abs(val - pi).max():.1e}")
    np.testing.assert_allclose(val, pi, atol=1e-10)
print("합격 기준 1: 다섯 방법이 1e-10 이내로 일치")

in-degree만으로 순위가 설명되는지 본다. 노드 2와 4는 둘 다 in-degree 2인데 $\pi$는 0.072 vs 0.348이다.

In [ ]:
in_deg, out_deg = A.sum(axis=0), A.sum(axis=1)
in_share = in_deg / in_deg.sum()
fig, ax = plt.subplots()
w = 0.38
ax.bar(np.arange(n) - w / 2, pi, w, label="PageRank pi_i (d = 0.85)")
ax.bar(np.arange(n) + w / 2, in_share, w, label="in-degree share", alpha=0.7)
ax.set(xlabel="node", ylabel="probability", title="PageRank vs in-degree share, per node")
ax.legend(); plt.show()

rank = np.empty(n, int); rank[np.argsort(-pi)] = np.arange(1, n + 1)
rows_md = ["| node | out | in | pi | rank |", "|---|---|---|---|---|"]
rows_md += [f"| {i} | {int(out_deg[i])} | {int(in_deg[i])} | {pi[i]:.6f} | {rank[i]} |" for i in range(n)]
Markdown("\n".join(rows_md))

표에서 $\pi_1 = \pi_3$이 정확한 동률임을 확인하세요(둘 다 노드 0에서만 1/3씩 받는다). 순위표는 `argsort`가 동률을 임의로 깬 것이므로 1과 3의 순위 차이는 의미가 없다.

### Task 3 — 거듭제곱 반복의 오차와 스펙트럼 (01e 재사용)

균등 $x_0$에서 $\|x_k-\pi\|_1$을 100회 기록하고 정리 2의 상계 $d^k\|x_0-\pi\|_1$과 겹친다. $d\in\{0.5, 0.7, 0.85, 0.95\}$를 모두 그려 기울기가 정확히 $\log d$임을 본다.

In [ ]:
n_iter, d_list = 100, [0.5, 0.7, 0.85, 0.95]
x0 = np.full(n, 1.0 / n)
errs, xs = power_iteration(G, x0, n_iter, pi)          # d = 0.85
k = np.arange(n_iter + 1)

fig, ax = plt.subplots()
for dd in d_list:
    G_d, _ = google_matrix(A, dd)
    e_d, _ = power_iteration(G_d, x0, n_iter, stationary(G_d))
    line, = ax.semilogy(k, e_d, label=f"d = {dd}")
    ax.semilogy(k, dd ** k * e_d[0], "--", color=line.get_color(), lw=0.9)
ax.axhline(1e-6, color="0.5", lw=0.8)
ax.set(xlabel="iteration k", ylabel=r"$\|x_k - \pi\|_1$", ylim=(1e-12, 3),
       title="Power-iteration L1 error (solid) vs bound $d^k\\|x_0-\\pi\\|_1$ (dashed)")
ax.legend(); plt.show()

In [ ]:
ratios = errs[1:] / errs[:-1]
slope = np.polyfit(k[1:], np.log(errs[1:]), 1)[0]
k_star = int(np.flatnonzero(errs < 1e-6)[0])            # 처음으로 1e-6 아래로 내려가는 k (없으면 IndexError)
k_bound = int(np.ceil(np.log(1e-6 / errs[0]) / np.log(d)))
print(f"||x_0 - pi||_1 = {errs[0]:.4f},  errs[1:4] = {errs[1:4]}")
print(f"consecutive ratios k=20..30: min {ratios[19:30].min():.4f}, max {ratios[19:30].max():.4f}  (d = {d})")
print(f"least-squares rate exp(slope) = {np.exp(slope):.4f}")
print(f"first k with error < 1e-6: {k_star}   (contraction bound guarantees k <= {k_bound})")
assert np.all(errs[1:] <= d * errs[:-1] + 1e-12), "수축 정리 위반"
assert errs[k_star] < 1e-6 and k_star > 0 and k_star <= k_bound
assert np.all(errs <= 2 * d ** k + 1e-12)

매 반복 오차가 정확히 $d$배 근처로 줄어든다(비율 0.84–0.85). 이는 $|\lambda_2(G)| = d$이기 때문이며, 정리 3으로 확인한다: $G$의 고유값 절댓값(1 제외)이 $d\times$ $S$의 고유값 절댓값(1 하나 제외)과 같아야 한다.

In [ ]:
abs_S, abs_G = abs_eigs_sorted(S), abs_eigs_sorted(G)
print("|eig(S)| sorted:", abs_S)
print("|eig(G)| sorted:", abs_G)
print("eig(S):", np.sort_complex(np.linalg.eigvals(S)).round(4))
print("eig(G):", np.sort_complex(np.linalg.eigvals(G)).round(4))
lambda2_G = abs_G[1]
np.testing.assert_allclose(lambda2_G, d, atol=1e-10)                 # |lambda_2(G)| = d
np.testing.assert_allclose(abs_G[1:], d * abs_S[1:], atol=1e-8)      # |lambda_k(G)| = d |lambda_k(S)|, k >= 2
print(f"합격 기준 2: |lambda_2(G)| = {lambda2_G:.10f} = d")

$S$의 고유값 $-1$은 주기 2인 닫힌 류 $\{4,5\}$에서 오고, $G$에서 $-0.85$가 된다. 행별 TV 거리의 최댓값 $\max_x \mathrm{TV}(G^n(x,\cdot),\pi)$도 정리 2의 $d^n$ 아래에 있어야 한다.

In [ ]:
for m in (20, 40, 60):
    G_m = np.linalg.matrix_power(G, m)
    tv_max = max(tv_distance(G_m[x], pi) for x in range(n))
    print(f"n = {m:2d}: max_x TV(G^n(x,.), pi) = {tv_max:.3e}   bound d^n = {d ** m:.3e}")
    assert tv_max <= d ** m

# 랭크 싱크가 없는 변형(5 -> 4 링크 제거): |lambda_2(G)| < d 가 된다
A_ns = A.copy(); A_ns[5, 4] = 0
G_ns, S_ns = google_matrix(A_ns, d)
print(f"no-sink variant: |lambda_2(S)| = {abs_eigs_sorted(S_ns)[1]:.4f}, "
      f"|lambda_2(G)| = {abs_eigs_sorted(G_ns)[1]:.4f} = d * |lambda_2(S)|  (< d = {d})")

### Task 4 — 랜덤 서퍼 몬테카를로 (00b/01c 재사용)

독립 서퍼 `N_PATHS`명을 노드 0에서 출발시켜 60걸음 뒤 위치를 본다. 종착 위치는 iid 표본이므로 `mc_proportion`이 SE를 준다. 편향은 $\mathrm{TV}(G^{60}(0,\cdot),\pi) \le d^{60} = 5.8\times10^{-5}$로 SE보다 두 자릿수 작다.

In [ ]:
n_steps_dist = 60
paths60 = simulate_chain(G, 0, n_steps_dist, rng, n_paths=N_PATHS)
final = paths60[:, -1]
occ = [mc_proportion(final == i) for i in range(n)]
occ_mean = np.array([r.mean for r in occ]); occ_se = np.array([r.se for r in occ])
time_avg = empirical_distribution(paths60, n, burn_in=20)         # 시간 평균: SE 없음(자기상관)

fig, ax = plt.subplots()
w = 0.28
ax.bar(np.arange(n) - w, pi, w, label="exact PageRank pi")
ax.bar(np.arange(n), occ_mean, w, yerr=1.96 * occ_se, capsize=3, label=f"MC: position after {n_steps_dist} steps (95% CI)")
ax.bar(np.arange(n) + w, time_avg, w, alpha=0.6, label="time average (burn-in 20, no SE)")
ax.set(xlabel="node", ylabel="probability", title=f"Random-surfer occupancy, {N_PATHS} surfers from node 0")
ax.legend(); plt.show()
for i in range(n):
    print(f"node {i}: MC {occ[i]}   exact {pi[i]:.4f}   z = {(occ_mean[i] - pi[i]) / occ_se[i]:+.2f}")

이제 **도달시간**과 **재귀시간**. 노드 0에서 노드 4에 처음 닿는 기대 시간 $E_0[T_4]$는 01c의 기본행렬(노드 4를 흡수 상태로 만든 사슬)로, 재귀시간 $E_0[T_0^+]$는 Kac 공식 $1/\pi_0$(01d/04c)로 정확히 구한다. 400걸음 안에 못 닿을 확률 $P_0(T_4>400)$(노드 4를 뺀 부분행렬 $Q$로 $Q^{400}$의 행 0 합)은 $\approx 3\times10^{-40}$으로 무시할 수 있다 — 아래에서 계산하고 `np.isinf` 개수가 0인지도 확인한다.

In [ ]:
n_steps_hit = 400
paths400 = simulate_chain(G, 0, n_steps_hit, rng, n_paths=N_PATHS)
T4 = hitting_times(paths400, 4)
T0 = first_return_times(paths400, 0)
assert np.isinf(T4).sum() == 0 and np.isinf(T0).sum() == 0
hit_mc, ret_mc = mc_estimate(T4), mc_estimate(T0)

T_set = [0, 1, 2, 3, 5, 6]                                                       # 노드 4를 흡수 상태로
hit_exact = absorption(G, transient=T_set, absorbing=[4])["t"][0]                # 01c 기본행렬
Q4 = G[np.ix_(T_set, T_set)]
print(f"P_0(T_4 > {n_steps_hit}) = {np.linalg.matrix_power(Q4, n_steps_hit)[0].sum():.1e}   (truncation bias negligible)")
ret_exact = 1.0 / pi[0]                                                          # Kac
print(f"E_0[T_4]:   MC {hit_mc}   exact {hit_exact:.6f}   (sd of T_4 = {T4.std():.2f})")
print(f"E_0[T_0^+]: MC {ret_mc}   exact {ret_exact:.6f}   (sd of T_0^+ = {T0.std():.2f})")

### Task 5 — 민감도: damping, 개인화, 매달린 노드 누수

$d$를 $0.1$부터 $0.99$까지 바꾸며 $\pi_i(d)$ 곡선을 그린다. 정리 4의 노이만 급수로 보면 $d\to0$은 '텔레포트 직후'($\pi\to v$), $d\to1$은 '텔레포트 없이 한없이 걷기'(랭크 싱크에 갇힘)다.

In [ ]:
d_grid = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 0.99])
pi_d = np.array([stationary(google_matrix(A, dd)[0]) for dd in d_grid])

fig, ax = plt.subplots()
for i in range(n):
    ax.plot(d_grid, pi_d[:, i], marker="o", ms=3, label=f"node {i}")
ax.axhline(1 / n, color="0.5", ls=":", lw=0.8)
ax.set(xlabel="damping d", ylabel=r"$\pi_i(d)$", title="PageRank of each node vs damping d")
ax.legend(ncol=4, fontsize=8)
ins = ax.inset_axes([0.12, 0.45, 0.32, 0.32])
ins.plot(d_grid, pi_d[:, 4] + pi_d[:, 5], color="tab:red")
ins.set(title="mass in sink {4,5}", ylim=(0, 1.05)); ins.tick_params(labelsize=7)
plt.show()
print("rank order (best first) for each d:")
for dd, p in zip(d_grid, pi_d):
    print(f"  d = {dd:4.2f}: {np.argsort(-p)}   pi_4 + pi_5 = {p[4] + p[5]:.3f}")

1·2위(4, 5)는 $d \ge 0.3$에서 고정이고, $d \le 0.2$에서는 균등에 가까워져 허브 0이 5를 앞선다. 그림에서 노드 1의 곡선은 노드 3의 곡선 아래에 완전히 숨어 있다 — 모든 $d$에서 $\pi_1=\pi_3$인 정확한 동률이며, 순위표의 1·3 순서는 `argsort`가 임의로 깬 것이다. 합격 기준 4: $d = 0.99$에서 싱크 질량 $\approx 0.97$, $d=0.1$에서 모두 $1/7$ 근처.

**개인화(personalized) PageRank.** 텔레포트 분포 $v$를 $e_0$, $e_1$로 바꾸면 순위가 바뀌지만 정리 3에 따라 고유값은 그대로다.

In [ ]:
print(f"{'v':10s} {'pi':64s} rank         |lambda_2(G)|")
for name, v in [("uniform", v_unif), ("e_0", np.eye(n)[0]), ("e_1", np.eye(n)[1])]:
    G_v, _ = google_matrix(A, d, v)
    p_v = stationary(G_v)
    np.testing.assert_allclose(p_v, linear_form_pagerank(S, d, v), atol=1e-10)   # 정리 4, 일반 v
    print(f"{name:10s} {str(p_v):64s} {np.argsort(-p_v)}  {abs_eigs_sorted(G_v)[1]:.6f}")

**매달린 노드 누수 실험.** 매달린 행을 고치지 않고 0으로 두면 $P_0 = dS_0 + (1-d)/n$의 행합이 노드 6에서 $1-d$뿐이라, 매 반복 노드 6에 있던 질량의 $d$배가 사라진다. 그리고 $d=1$(텔레포트 없음)이면 $G = S$가 되어 정리 1의 가정이 깨진다.

In [ ]:
S0 = np.where(out_deg[:, None] > 0, A / np.where(out_deg > 0, out_deg, 1.0)[:, None], 0.0)   # 매달린 행 = 0
P0 = d * S0 + (1 - d) / n
x = x0.copy()
for _ in range(300):
    x = x @ P0
leak_mass = x.sum()
print(f"row sums of P0: {P0.sum(axis=1)}")
print(f"total mass after 300 iterations = {leak_mass:.3e}   (합격 기준 5)")
print(f"normalized leaked vector = {x / leak_mass}\ntrue pi                  = {pi}   TV = {tv_distance(x / leak_mass, pi):.4f}")
assert leak_mass < 1e-4

pi_S = stationary(S)                       # d = 1: G = S
_, xs_S = power_iteration(S, x0, 201, pi_S)
print(f"\nd = 1: stationary(S) = {pi_S}")
print(f"       x_200 = {xs_S[200]}\n       x_201 = {xs_S[201]}   (period-2 oscillation on nodes 4, 5)")
np.testing.assert_allclose(pi_S, [0, 0, 0, 0, 0.5, 0.5, 0], atol=1e-10)

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
pred_top = predictions["pi_max"] if predictions.get("pi_top_node") == 4 else None
rows = [
    {"name": "pi_4 (top node) after 60 steps", "predicted": pred_top, "estimate": occ[4], "exact": pi[4]},
    {"name": "pi_0 after 60 steps", "predicted": predictions["pi_node0"], "estimate": occ[0], "exact": pi[0]},
    {"name": "pi_6 (dangling node) after 60 steps", "predicted": None, "estimate": occ[6], "exact": pi[6]},
    {"name": "pi_5 after 60 steps", "predicted": None, "estimate": occ[5], "exact": pi[5]},
    {"name": "E_0[T_4] hitting time 0 -> 4", "predicted": predictions["hit_time_0_to_4"], "estimate": hit_mc, "exact": hit_exact},
    {"name": "E_0[T_0^+] return time to 0 (= 1/pi_0)", "predicted": predictions["return_time_0"], "estimate": ret_mc, "exact": ret_exact},
]
Markdown(compare_table(rows))

- $\pi_4, \pi_0, \pi_6, \pi_5$: 어긋나면 `simulate_chain`이 $G$를 따르지 않거나(매달린 행 처리) 60걸음이 부족한 것 — 후자는 정리 2로 배제된다.
- $E_0[T_4]$: 어긋나면 기본행렬의 일시 상태 집합이 잘못됐거나 400걸음 절단이 편향을 만든 것.
- $E_0[T_0^+]$: 어긋나면 Kac 공식 $1/\pi_0$이 틀렸거나 `first_return_times`가 $t=0$을 잘못 세는 것.

결정론적 항목(합격 기준 1, 2, 5)은 이미 위 셀들의 `assert_allclose`/`assert`로 검사했다. 마지막으로 MC 대 닫힌 형식만 4 SE로 판정한다.

In [ ]:
for i in range(n):
    assert_close(occ[i], pi[i], k=4, name=f"pi_{i} after 60 steps")
assert_close(hit_mc, hit_exact, k=4, name="E_0[T_4]")
assert_close(ret_mc, ret_exact, k=4, name="E_0[T_0^+]")
print("합격 기준 3: 모든 노드에서 4 SE 이내 — 모든 검사 통과")

## 6. 연습문제

### E1 계산

3개 노드 그래프 0→{1,2}, 1→{0}, 2→(매달림)에서 $d=0.85$, 균등 텔레포트일 때 PageRank를 손으로 구하라. 힌트: 노드 1과 2는 같은 곳에서 같은 비율로 받으므로 $\pi_1=\pi_2=q$, $\pi_0 = 1-2q$. 또 3-사이클 0→1→2→0에서는 $|\lambda_2(G)|$가 얼마인지 정리 3으로 답하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$S = \begin{pmatrix}0&\tfrac12&\tfrac12\\ 1&0&0\\ \tfrac13&\tfrac13&\tfrac13\end{pmatrix}$ (노드 2는 매달려 균등). 텔레포트 항은 $(1-d)/3 = 0.05$. 노드 1의 방정식:

$$q = 0.05 + 0.85\Big(\tfrac{1-2q}{2} + \tfrac{q}{3}\Big) = 0.475 - 0.5667\,q \quad\Rightarrow\quad q = \frac{0.475}{1.5667} = 0.30319,\qquad \pi_0 = 1 - 2q = 0.39362 .$$

3-사이클: $S$는 순환행렬이라 고유값이 1의 세제곱근(절댓값 모두 1) → 정리 3에 의해 $|\lambda_2(G)| = d = 0.85$. 대칭이므로 $\pi = (\tfrac13, \tfrac13, \tfrac13)$.

```python
A3 = np.array([[0, 1, 1], [1, 0, 0], [0, 0, 0]])
q = 0.475 / (1 + 0.85 - 0.85 / 3)          # q = 0.05 + 0.85((1-2q)/2 + q/3)
pi_hand = np.array([1 - 2 * q, q, q])
print("hand:", pi_hand, " spkit:", pagerank(A3, 0.85))
np.testing.assert_allclose(pi_hand, pagerank(A3, 0.85), atol=1e-12)
A_cyc = np.array([[0, 1, 0], [0, 0, 1], [1, 0, 0]])
G_cyc, S_cyc = google_matrix(A_cyc, 0.85)
print("3-cycle |eig(S)|:", abs_eigs_sorted(S_cyc), " |eig(G)|:", abs_eigs_sorted(G_cyc), " pi:", stationary(G_cyc))
# hand: [0.39362 0.30319 0.30319]  spkit: [0.39362 0.30319 0.30319]
# 3-cycle |eig(S)|: [1. 1. 1.]  |eig(G)|: [1.   0.85 0.85]  pi: [0.33333 0.33333 0.33333]
```

</details>

### E2 유도 후 시뮬레이션 검증

$E_x[T_x^+] = 1/\pi_x$ (Kac)를 01d/04c의 재생 논법으로 유도한 뒤, 노드 4와 노드 0에 대해 `simulate_chain`(x0 = 해당 노드, n_paths = `N_PATHS`, n_steps = 200)으로 첫 귀환시간 평균을 추정하고 `assert_close(k=4)`로 검증하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**유도.** 노드 $x$의 방문 시각들은 재생 과정이다(강마르코프: 방문할 때마다 사슬이 $x$에서 새로 시작). 사이클 하나(귀환 한 번)당 '방문 1회'를 보상으로 주면 재생-보상 정리에 의해 장기 방문 빈도 $= 1/E_x[T_x^+]$. 에르고딕 정리(01d의 세 번째 방법)로 이 빈도는 $\pi_x$이므로 $E_x[T_x^+] = 1/\pi_x$.

**시뮬레이션.** 경로가 $x$에서 출발하므로 열 1부터 첫 $x$ 방문 인덱스가 $T_x^+$다(`first_return_times`). 노드 4는 평균 $1/0.34827 = 2.871$(정확한 sd ≈ 2.6), 노드 0은 $9.678$(정확한 sd ≈ 13.0; 표본 sd는 이 근처에서 요동한다). $P_0(T_0^+ > 200)\approx 1.0\times10^{-6}$이므로 200걸음 절단은 무해하지만 `np.isinf` 개수가 0인지 확인한다.

```python
for x in (4, 0):
    paths_x = simulate_chain(G, x, 200, rng, n_paths=N_PATHS)
    T = first_return_times(paths_x, x)
    assert np.isinf(T).sum() == 0, "200걸음 안에 못 돌아온 경로가 있음"
    est = mc_estimate(T)
    print(f"E_{x}[T_{x}^+]: MC {est}   exact 1/pi_{x} = {1 / pi[x]:.6f}   sd = {T.std():.2f}")
    assert_close(est, 1 / pi[x], k=4, name=f"E_{x}[T_{x}^+]")
# E_4[T_4^+]: MC 2.8060 ± 0.0539 (n=2000)   exact 1/pi_4 = 2.871338   sd = 2.41
# E_0[T_0^+]: MC 9.6940 ± 0.2874 (n=2000)   exact 1/pi_0 = 9.677628   sd = 12.85
```

</details>

### E3 가정을 깨보기

(a) $d=1$(텔레포트 없음)로 두고 균등 초기벡터에서 거듭제곱 반복을 200회 돌려라 — 수렴하는가? `stationary(S)`는 무엇을 주는가? (b) 매달린 노드 행을 0으로 둔 채 $d=0.85$로 300회 반복하면 벡터의 총합은 얼마가 되는가? 정규화하면 참 $\pi$와 같은가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 수렴하지 않는다. 질량이 $\{4,5\}$로 모두 흘러든 뒤 두 노드 사이에서 주기 2로 진동한다: 200회 후 $x_4 = 0.43478,\ x_5 = 0.56522$, 201회 후 서로 바뀐다. 정리 1의 가정 $d<1$을 위반해 기약성(닫힌 류 $\{4,5\}$ 밖의 상태는 일시)과 비주기성(주기 2)을 모두 잃었다. `stationary(S)` $= [0,0,0,0,\tfrac12,\tfrac12,0]$ — 닫힌 류 하나에만 실린 해다. 이 그래프에서는 닫힌 류가 하나뿐이라 우연히 유일하지만, 선형계 풀이는 어떤 경우든 특정 해 하나를 돌려줄 뿐 유일성을 증명하지 않는다(01b). $\|x_k - \pi_S\|_1$은 $0.1304$에 머문다.

(b) 총합 $6.754\times10^{-5}$ — 매 반복 매달린 노드에 있던 질량 $d\,x_6$이 사라진다. 정규화 벡터 $[0.092,0.049,0.063,0.049,0.367,0.344,0.036] \ne \pi$ (노드 6이 0.036 vs 0.042). 수축 정리는 행합 1을 요구하며, 행합 1이 아니면 '정상분포'라는 개념 자체가 정의되지 않는다.

```python
errs_S, xs_S = power_iteration(S, np.full(n, 1 / n), 201, stationary(S))
print("(a) x_200 =", xs_S[200], "\n    x_201 =", xs_S[201])
print("    stationary(S) =", stationary(S), "  errs tail:", errs_S[-3:])
S0 = np.where(out_deg[:, None] > 0, A / np.where(out_deg > 0, out_deg, 1.0)[:, None], 0.0)
P0 = d * S0 + (1 - d) / n
x = np.full(n, 1 / n)
for _ in range(300):
    x = x @ P0
print(f"(b) total mass after 300 iters = {x.sum():.3e};  normalized = {x / x.sum()}\n    true pi = {pi}")
# (a) x_200 = [0. 0. 0. 0. 0.43478 0.56522 0.]   x_201 = [0. 0. 0. 0. 0.56522 0.43478 0.]
#     stationary(S) = [0. 0. 0. 0. 0.5 0.5 0.]   errs tail: [0.13043 0.13043 0.13043]
# (b) total mass after 300 iters = 6.754e-05;  normalized = [0.09202 0.04902 0.06335 0.04902 0.36653 0.34362 0.03645]
```

</details>

### 스트레치 S1–S3 (선택)

**S1 (개인화 PageRank).** $v = e_0$, $v = e_1$, $v$ = $\{0,1,2\}$에 균등, 각각에 대해 $\pi$와 순위를 구하고, 정리 3이 $v$에 무관하게 $|\lambda_2(G)| = d$를 주는지 수치로 확인하라.

**S2 (규모 확장).** $n = 2000$ 노드, 각 노드가 무작위 3개 노드로 링크하고 5%는 매달린 무작위 그래프를 `scipy.sparse.csr_matrix`로 만들고, 희소 행렬-벡터 곱만으로 거듭제곱 반복을 구현하라(매달린 질량은 스칼라로 모아 균등 분배). 반복 횟수가 $n$과 무관하게 $\le 80$임을 확인하고 밀집 `stationary`와 비교하라.

**S3 (랭크 싱크 제거).** 5→4 링크를 지운 변형(5도 매달림)에서 $|\lambda_2(S)|$, $|\lambda_2(G)|$, 거듭제곱 반복의 실제 감쇠율을 측정하고 정리 2(상계 $d$)와 정리 3(정확한 값 $d|\lambda_2(S)|$)을 대조하라.

In [ ]:
# 여기에 풀이 (S1–S3)

<details><summary>정답</summary>

**S1.** `google_matrix(A, 0.85, v)`로 계산한다. $v=e_0$: $\pi = [0.2572, 0.0758, 0.0973, 0.0758, 0.2521, 0.2173, 0.0245]$ → 노드 0이 1위. $v=e_1$: $[0.1597, 0.2032, 0.1108, 0.0532, 0.2160, 0.1916, 0.0655]$ → 4, 1, 5 순. $v$가 어떤 확률벡터든 $Q^{-1}GQ$의 블록 구조는 같으므로 고유값은 $v$와 무관 — 세 경우 모두 $|\lambda_2(G)| = 0.85$.

```python
for name, v in [("e_0", np.eye(n)[0]), ("e_1", np.eye(n)[1]), ("unif{0,1,2}", np.r_[1, 1, 1, 0, 0, 0, 0] / 3)]:
    G_v, _ = google_matrix(A, 0.85, v); p_v = stationary(G_v)
    print(f"v={name:12s} pi={p_v}  rank={np.argsort(-p_v)}  |lambda_2|={abs_eigs_sorted(G_v)[1]:.4f}")
```

**S2.** $x_{k+1} = d\,x S_{\text{sparse}} + \big(d\sum_{i\ \text{dangling}}x_i + (1-d)\big)/n$. $10^{-10}$ 수렴에 약 31회(수십 ms); 밀집 `stationary`는 $2000\times2000$ solve라 수백 ms. 두 결과는 $10^{-8}$ 이내로 같다.

```python
import scipy.sparse as sp
def sparse_pagerank(A_csr, d, tol=1e-10, max_iter=1000):
    n = A_csr.shape[0]
    out = np.asarray(A_csr.sum(axis=1)).ravel(); dangling = out == 0
    S_csr = sp.diags(np.where(dangling, 0.0, 1.0 / np.where(dangling, 1.0, out))) @ A_csr
    x = np.full(n, 1.0 / n)
    for k in range(1, max_iter + 1):
        x_new = d * (S_csr.T @ x) + (d * x[dangling].sum() + (1 - d)) / n
        if np.abs(x_new - x).sum() < tol:
            return x_new, k
        x = x_new
    return x, max_iter
n_big = 2000
targets = rng.integers(0, n_big, size=(n_big, 3))
A_big = np.zeros((n_big, n_big)); A_big[np.repeat(np.arange(n_big), 3), targets.ravel()] = 1
A_big[rng.random(n_big) < 0.05] = 0          # 5%는 매달린 노드
x_sp, k_sp = sparse_pagerank(sp.csr_matrix(A_big), 0.85)
pi_big = stationary(google_matrix(A_big, 0.85)[0])
print(f"sparse: {k_sp} iterations;  max|diff| vs dense = {np.abs(x_sp - pi_big).max():.1e}")
assert k_sp <= 80
np.testing.assert_allclose(x_sp, pi_big, atol=1e-8)
# sparse: 31 iterations;  max|diff| vs dense = 1.9e-13
```

**S3.** 변형 그래프: $|\lambda_2(S)| = 0.3975$, $|\lambda_2(G)| = 0.3379 = 0.85\times0.3975$. 오차 비율은 $\lambda_2$가 복소수라 반복마다 요동하지만(0.22–0.54) 기하평균은 $\approx 0.34$로, 수축 상계 0.85보다 훨씬 빠르다. $\pi = [0.2097, 0.1134, 0.1455, 0.1134, 0.1503, 0.1817, 0.0861]$ — 싱크가 사라지자 허브 0이 1위가 된다.

```python
A_ns = A.copy(); A_ns[5, 4] = 0
G_ns, S_ns = google_matrix(A_ns, 0.85); pi_ns = stationary(G_ns)
errs_ns, _ = power_iteration(G_ns, np.full(n, 1 / n), 40, pi_ns)
print(f"|lambda_2(S)| = {abs_eigs_sorted(S_ns)[1]:.5f}, |lambda_2(G)| = {abs_eigs_sorted(G_ns)[1]:.5f}")
print("error ratios k=10..15:", errs_ns[11:16] / errs_ns[10:15], "\npi =", pi_ns)
# |lambda_2(S)| = 0.39750, |lambda_2(G)| = 0.33788
# error ratios k=10..15: [0.23656 0.22127 0.5433  0.49174 0.35211]
```

</details>

## 7. 정리

1. PageRank는 구글 행렬 $G = dS + (1-d)\mathbf{1}v^\top$의 정상분포다; 매달린 노드는 균등 행으로 고치고, 텔레포트가 기약·비주기성을 강제한다.
2. $\pi$는 선형계·고유벡터·거듭제곱·$(1-d)v^\top(I-dS)^{-1}$ 네 가지로 같은 답 $[0.1033, 0.0559, 0.0717, 0.0559, 0.3483, 0.3226, 0.0424]$를 준다.
3. $L^1$ 수축 정리: 매 반복 오차가 $d$배 이하 — 수렴 속도는 그래프가 아니라 $d$가 정한다($10^{-6}$까지 72회, 상계 84회). 스펙트럼: $\lambda_k(G) = d\lambda_k(S)$.
4. 독립 서퍼의 종착 위치는 iid 표본이라 SE가 나오고 $\pi$와 4 SE 이내로 맞는다; 도달시간 5.52와 재귀시간 $1/\pi_0 = 9.68$도 기본행렬·Kac 공식과 일치한다.
5. $d\to1$이면 랭크 싱크 $\{4,5\}$가 모든 질량을 가두고 반복은 진동한다; 매달린 노드를 방치하면 질량이 샌다 — 가정을 깨면 정리도 깨진다.

**trap 카드.** Q: PageRank의 거듭제곱 반복은 그래프가 클수록 더 많이 돌려야 하는가? → A: 아니다. 수축 정리에 의해 $\|x_k-\pi\|_1 \le 2d^k$ — 반복 횟수는 $d$와 원하는 정밀도만으로 정해진다($d=0.85$, $10^{-6}$: 일반 상계 $2d^k$로 90회 이하; 본 그래프의 균등 $x_0$에서는 상계 84회, 실제 72회). 그래프 구조는 $|\lambda_2(G)| = d|\lambda_2(S)| \le d$를 통해 더 빠르게 만들 뿐이다.

**다음 노트북: 02a.** 모듈 01의 이산시간 사슬을 마쳤다. 다음은 시간을 연속으로 바꾸는 재료 — 지수분포의 무기억성과 경쟁하는 시계 — 이며 이는 푸아송 과정(02)과 CTMC(03)의 토대다.

log/progress.md 한 줄 템플릿: `- YYYY-MM-DD C1 PageRank — 합격 기준 1–5 통과 (N_PATHS=…, 최대 |z|=…), 막힌 곳: …, 다음: 02a`